In [1]:
import scanpy as sc
import functools
import os
import sys
import traceback
from typing import Dict, Literal, Optional, Tuple
import pickle
import cellflow
from cellflow import preprocessing as cfpp
import scanpy as sc
import numpy as np
import functools
from ott.solvers import utils as solver_utils
import optax
from omegaconf import OmegaConf
from typing import NamedTuple, Any
import hydra
import wandb
import pandas as pd
import time
import anndata as ad

from numpy.typing import ArrayLike


/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [2]:
def add_embeddings(adata):
    with open('/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/gene_nargab.pkl', 'rb') as f:
        nargab_emb = pickle.load(f)
    adata.uns["nargab"] = nargab_emb

    with open('/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/gene_mix.pkl', 'rb') as f:
        mix_emb = pickle.load(f)
    adata.uns["mix"] = mix_emb

In [3]:

split = 0
DATA_DIR = "/lustre/groups/ml01/workspace/ot_perturbation/data/norman_new"

adata_train_path = os.path.join(DATA_DIR, f"adata_train_pca_50_split_{split}.h5ad")
adata_test_path = os.path.join(DATA_DIR, f"adata_test_pca_50_split_{split}.h5ad")
adata_ood_path = os.path.join(DATA_DIR, f"adata_ood_pca_50_split_{split}.h5ad")

adata_train = sc.read_h5ad(adata_train_path)
adata_test = sc.read_h5ad(adata_test_path)
adata_ood = sc.read_h5ad(adata_ood_path)

add_embeddings(adata_train)
add_embeddings(adata_test)
add_embeddings(adata_ood)


adata_ood_single = adata_ood[(adata_ood.obs["kategory"] == "ctrl") | (adata_ood.obs["subgroup"]=="single")]
adata_ood_double_seen_0 = adata_ood[(adata_ood.obs["kategory"] == "ctrl") | (adata_ood.obs["subgroup"]=="double_seen_0")]
adata_ood_double_seen_1 = adata_ood[(adata_ood.obs["kategory"] == "ctrl") | (adata_ood.obs["subgroup"]=="double_seen_1")]
adata_ood_double_seen_2 = adata_ood[(adata_ood.obs["kategory"] == "ctrl") | (adata_ood.obs["subgroup"]=="double_seen_2")]


In [4]:
adata_train.uns.keys()

dict_keys(['esm2', 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'pca', 'rank_genes_groups_cov_all', 'top_non_dropout_de_20', 'top_non_zero_de_20', 'nargab', 'mix'])

In [13]:

cf = cellflow.model.CellFlow(adata_train, solver="otfm")

# Prepare the training data and perturbation conditions
perturbation_covariates = {"target_gene": ("gene_1", "gene_2")}
perturbation_covariate_reps = {"target_gene": "esm2"}

cf.prepare_data(
    sample_rep="X_pca",
    control_key="control",
    perturbation_covariates=perturbation_covariates,
    perturbation_covariate_reps=perturbation_covariate_reps,
    sample_covariates=None,
    sample_covariate_reps=None,
    split_covariates=None
)

match_fn = functools.partial(
    solver_utils.match_linear,
    epsilon=1.0,
    scale_cost="mean",
)

[########################################] | 100% Completed | 156.93 ms
[########################################] | 100% Completed | 102.48 ms
[########################################] | 100% Completed | 103.14 ms


In [14]:
adata_vals = {}
adata_ctrl = adata_ood_single[adata_ood_single.obs["condition"]=="ctrl"]
for cond in adata_ood_single.obs["condition"].unique():
    if cond == "ctrl":
        continue
    adata_pert = adata_ood_single[adata_ood_single.obs["condition"]==cond]
    cdata = ad.concat((adata_ctrl, adata_pert))
    cdata.uns = adata_train.uns.copy()
    adata_vals[cond] = cdata


In [15]:


optimizer = optax.MultiSteps(optax.adam(5e-5), 20)
probability_path= {"constant_noise": 0.5}

layers_before_pool = {"target_gene": {"layer_type": "mlp", "dim": [1024, 1024]}}
layers_after_pool = {"layer_type": "mlp", "dim": [1024, 1024]}

solver_kwargs = {"ema": 1.0}
time_max_period = 0
time_max_period = time_max_period if time_max_period>0 else None

# Prepare the model
cf.prepare_model(
    condition_mode="deterministic",
    regularization=0.1,
    pooling="mean",
    layers_before_pool=layers_before_pool,
    layers_after_pool=layers_after_pool,
    condition_embedding_dim=256,
    cond_output_dropout=0.5,
    time_freqs=1024,
    time_max_period=time_max_period,
    time_encoder_dims=[1024, 1024, 1024],
    time_encoder_dropout=0.0,
    hidden_dims=[2048, 2048, 2048],
    hidden_dropout=0.0,
    conditioning="film",
    decoder_dims=[2048, 2048, 2048],
    decoder_dropout=0.0,
    probability_path=probability_path,
    solver_kwargs=solver_kwargs,
    match_fn=match_fn,
    optimizer=optimizer,
    layer_norm_before_concatenation=False,
    linear_projection_before_concatenation=False,
)


for cond, bdata in adata_vals.items():
    cf.prepare_validation_data(
        bdata,
        name=cond,
        n_conditions_on_log_iteration=None,
        n_conditions_on_train_end=None,
    )



[########################################] | 100% Completed | 102.03 ms
[########################################] | 100% Completed | 101.75 ms
[########################################] | 100% Completed | 100.93 ms
[########################################] | 100% Completed | 101.47 ms
[########################################] | 100% Completed | 101.63 ms
[########################################] | 100% Completed | 100.92 ms
[########################################] | 100% Completed | 101.04 ms
[########################################] | 100% Completed | 102.14 ms
[########################################] | 100% Completed | 101.34 ms
[########################################] | 100% Completed | 101.15 ms
[########################################] | 100% Completed | 101.93 ms
[########################################] | 100% Completed | 100.79 ms
[########################################] | 100% Completed | 102.14 ms
[########################################] | 100% Completed | 10

In [16]:
from cellflow.training import ComputationCallback

In [17]:
class OutputPredsCallback(ComputationCallback):
    def __init__(self):
        self.preds_during_training = {}
        self.preds_at_end = {}

    def on_train_begin(self):
        pass

    def on_log_iteration(self, valid_source_data, valid_true_data, valid_pred_data, solver):
        self.preds_during_training = valid_pred_data
        return {"pred_data_on_log_iteration": valid_pred_data}

    def on_train_end(self, valid_source_data, valid_true_data, valid_pred_data, solver):
        self.preds_at_end = valid_pred_data
        return {"pred_data_on_train_end": valid_pred_data}

In [ ]:
#metrics_callback = cellflow.training.Metrics(metrics=["r_squared", "mmd", "e_distance"])
opc = OutputPredsCallback()
decoded_metrics_callback = cellflow.training.PCADecodedMetrics(ref_adata=adata_train, metrics=["r_squared"])
wandb_callback = cellflow.training.WandbLogger(
    project="norman_check", 
    out_dir="/lustre/groups/ml01/workspace/ot_perturbation/logging", 
    config={"foo": "bar"},
)
callbacks = [opc, decoded_metrics_callback, wandb_callback]

cf.train(
    num_iterations=20000,
    batch_size=1024,
    callbacks=callbacks,
    valid_freq=400000,
)

#if config_dict["training"]["save_model"]:
cf.save("/home/icb/dominik.klein/tmp", file_prefix="test_run2", overwrite=True)

#if config_dict["training"]["save_predictions"]:



wandb: WARNING Calling wandb.login() after wandb.init() has no effect.
 97%|█████████▋| 19302/20000 [04:50<00:09, 73.11it/s]

In [ ]:
metrics = {k:v for k,v in cf.trainer.training_logs.items() if k!="loss"}

In [ ]:
metrics

In [ ]:
adata_ood_ctrl = adata_ood[adata_ood.obs["control"]]
adata_ood_ctrl

In [ ]:
adata_ood_ctrl = adata_ood[adata_ood.obs["control"]]
covariate_data_test = adata_ood.obs.drop_duplicates(subset=["gene_1", "gene_2"])
preds_test = cf.predict(adata=adata_ood_ctrl, sample_rep="X_pca", condition_id_key="condition", covariate_data=covariate_data_test)
all_data = []
conditions = []

for condition, array in preds_test.items():
    all_data.append(array)
    conditions.extend([condition] * array.shape[0])

# Stack all data vertically to create a single array
all_data_array = np.vstack(all_data)

# Create a DataFrame for the .obs attribute
obs_data = pd.DataFrame({
    'condition': conditions
})

# Create the Anndata object
adata_test_result = ad.AnnData(X=np.empty((len(all_data_array), adata_train.shape[1])), obs=obs_data)
adata_test_result.obsm["X_pca_pred"] = all_data_array
cfpp.reconstruct_pca(query_adata=adata_test_result, use_rep="X_pca_pred", ref_adata=adata_train, layers_key_added="X_recon_pred")



In [ ]:
adata_save_path = os.path.join("/home/icb/dominik.klein/tmp", f"adata_test_with_predictions_{split}.h5ad")
adata_test_result.write(adata_save_path)


In [ ]:
adata_test_result.X = adata_test_result.layers["X_recon_pred"]

In [ ]:

import scanpy as sc
import jax
import os
from cellflow.metrics import compute_metrics, compute_mean_metrics, compute_metrics_fast
import cellflow.preprocessing as cfpp
import anndata as ad
import pandas as pd
import numpy as np
import sys
import pickle
import anndata as ad
import pandas as pd
from typing import Any, Tuple, Dict
import sys



ood_conds = adata_ood.obs.drop_duplicates()




adata_ref = ad.concat((adata_train, adata_test, adata_ood[adata_ood.obs["control"]==0]))
cfpp.centered_pca(adata_ref, n_comps=10)



adata_pred_ood = adata_test_result

cfpp.project_pca(query_adata=adata_pred_ood, ref_adata=adata_ref)
cfpp.project_pca(query_adata=adata_ood, ref_adata=adata_ref)
ood_data_target_encoded = {}
ood_data_target_decoded = {}
ood_data_target_encoded_predicted = {}
ood_data_target_decoded_predicted = {}
for cond in adata_ood.obs["condition"].cat.categories:
    if cond == "ctrl":
        continue
    ood_data_target_encoded[cond] = adata_ood[adata_ood.obs["condition"] == cond].obsm["X_pca"]
    ood_data_target_decoded[cond] = adata_ood[adata_ood.obs["condition"] == cond].X.toarray()
    ood_data_target_decoded_predicted[cond] = adata_pred_ood[adata_pred_ood.obs["condition"] == cond].X.toarray()
    ood_data_target_encoded_predicted[cond] = adata_pred_ood[adata_pred_ood.obs["condition"] == cond].obsm["X_pca"]


ood_deg_dict = {
    k: v
    for k, v in adata_train.uns["rank_genes_groups_cov_all"].items()
    if k in ood_data_target_decoded_predicted.keys()
}



ood_metrics_encoded = jax.tree_util.tree_map(
    compute_metrics_fast, ood_data_target_encoded, ood_data_target_encoded_predicted
)

ood_metrics_decoded = jax.tree_util.tree_map(
    compute_metrics_fast, ood_data_target_decoded, ood_data_target_decoded_predicted
)

#ood_metrics_deg = jax.tree_util.tree_map(
#    compute_metrics_fast, ood_deg_target_decoded, ood_deg_target_decoded_predicted
#)




In [ ]:
df_enc = pd.DataFrame.from_dict(ood_metrics_encoded,orient="index")#.to_csv(os.path.join(config_dict["training"]["out_dir"], f"{wandb.run.name}_{condition}.csv"))
df_dec = pd.DataFrame.from_dict(ood_metrics_decoded,orient="index")
#df_deg = pd.DataFrame.from_dict(ood_metrics_deg,orient="index")

for col in df_enc.columns:
    df_enc[f"encoded_ood_{col}"] = df_enc[col]
    del df_enc[col]

for col in df_dec.columns:
    df_dec[f"decoded_ood_{col}"] = df_dec[col]
    del df_dec[col]

#for col in df_deg.columns:
#    df_deg[f"deg_ood_{col}"] = df_deg[col]
#    del df_deg[col]

df_enc["condition"] = df_enc.index
df_all = pd.concat((df_enc, df_dec), axis=1)
df_cat = pd.concat((adata_train.obs.drop_duplicates(subset="condition"),adata_ood.obs.drop_duplicates(subset="condition")))
cond_to_cat = df_cat.set_index("condition")["subgroup"].to_dict()
df_all["subgroup"] = df_all["condition"].map(cond_to_cat)
df_all["model"] = "cellflow"



In [ ]:
len(metrics), len(ood_metrics_decoded)

In [ ]:
len(preds_test)

In [ ]:
{k: v["r_squared"] for k,v in ood_metrics_decoded.items() if "ctrl" in k}

In [ ]:
metrics

In [ ]:
adata_ood_ctrl, 

In [41]:
pca_1 = adata_ood_ctrl[adata_ood_ctrl.obs["condition"]=="ctrl"].obsm["X_pca"]


In [39]:
bdata = adata_vals['DUSP9+ctrl']
pca_2 = bdata[bdata.obs["condition"]=="ctrl"].obsm["X_pca"]

In [43]:
(pca_1 == pca_2).all()

True

In [17]:
df_s = df_all[df_all["subgroup"]=="single"]

In [18]:
df_s["decoded_ood_r_squared"].mean()

0.983873705069224

In [34]:
df_d2 = df_all[df_all["subgroup"]=="double_seen_2"]

In [35]:
df_d2["decoded_ood_r_squared"].mean()

0.9920696457227071

In [31]:
np.mean([v["r_squared"] for v in ood_metrics_decoded.values()])

0.977944330077305

In [ ]:
df_d2